In [67]:
import requests
from bs4 import BeautifulSoup
from app.crawler import SitemapCrawler
from app.parser import SiteParser
from app.pipeline import IngestionPipeline


In [62]:
# import importlib
# import app.parser as parser_module
# import app.crawler as crawler_module
# import  app.pipeline as pipeline_module
#
# importlib.reload(parser_module)
# importlib.reload(crawler_module)
# importlib.reload(pipeline_module)
# SiteParser = parser_module.SiteParser
# SitemapCrawler = crawler_module.SitemapCrawler
# IngestionPipeline = pipeline_module.IngestionPipeline

# Sitemap tests

In [3]:
url = "https://sk-family.ru"
response = requests.get(url, timeout=20)
html = response.text

print(response.status_code)
print(html[:500])

200
<!DOCTYPE HTML>
<html lang="ru-RU">
<head>
	<meta charset="UTF-8">
	<meta name='robots' content='index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1' />

	<!-- This site is optimized with the Yoast SEO plugin v26.6 - https://yoast.com/wordpress/plugins/seo/ -->
	<title>Компания по ремонту квартир в Москве</title>
	<meta name="description" content="Компания по ремонту квартир в Москве: Капитальный и дизайнерский ремонт в новостройке или вторичке. Индивидуальный ремонт


In [4]:
soup = BeautifulSoup(html, "html.parser")

In [5]:

print(soup.title)
print(soup.title.text)

<title>Компания по ремонту квартир в Москве</title>
Компания по ремонту квартир в Москве


In [6]:
h1 = soup.find("h1")

print(h1)
print(h1.text if h1 else "H1 not found")

<h1 style="text-align: center;">Ремонт квартир и дизайн интерьера  в Москве</h1>
Ремонт квартир и дизайн интерьера  в Москве


In [7]:
main = soup.find("main")

print(main)

<main class="l-main" id="page-content" itemprop="mainContentOfPage">
<section class="l-section wpb_row white height_small full_height valign_center width_full with_img"><div class="l-section-img" data-img-height="960" data-img-width="1440" role="img" style="background-image: url(https://sk-family.ru/wp-content/uploads/2026/03/bg_header.png);"></div><div class="l-section-h i-cf"><div class="g-cols vc_row via_grid cols_1 laptops-cols_inherit tablets-cols_inherit mobiles-cols_1 valign_top type_default stacking_default"><div class="wpb_column vc_column_container"><div class="vc_column-inner"><div class="wpb_text_column"><div class="wpb_wrapper"><h1 style="text-align: center;">Ремонт квартир и дизайн интерьера  в Москве</h1>
<div class="title_h2">От 15000₽/м²</div>
</div></div><div class="w-separator size_medium"></div><div class="w-hwrapper valign_middle stack_on_mobiles align_center"><div class="w-iconbox iconpos_top style_default color_primary align_center no_title"><div class="w-iconbox

# Parser Tests

## main page

In [17]:
parser = SiteParser(
    junk_selectors=[
        ".w-grid-json",
        ".l-popup",
        ".g-preloader",
        ".w-btn-wrapper",
    ],
    junk_phrases=[],
    min_text_length=200
)

In [18]:
doc = parser.parse_page(url, html)

In [19]:
print(doc.title)
print(doc.h1)
print(doc.headings[:10])
print(doc.text[:3000])
print(doc.metadata)

Компания по ремонту квартир в Москве
Ремонт квартир и дизайн интерьера в Москве
['Ремонт квартир и дизайн интерьера в Москве', 'Косметический ремонт', 'Капитальный ремонт', 'Дизайнерский ремонт', 'Посмотрите последние наши реализованные объекты', 'Наши преимущества', 'Виды ремонта', 'От 15000₽/м²', 'Прозрачный учет расходов', 'Фото и видеофиксация']
# Ремонт квартир и дизайн интерьера в Москве

### От 15000₽/м²

Официальный договор

Фиксированная смета

Гарантия до 3х лет

Соблюдение сроков

## Посмотрите последние наши реализованные объекты

## Наши преимущества

### Прозрачный учет расходов

Каждая операция по вашему ремонтному счету фиксируется и подтверждается детальным отчетом. Вы лично утверждаете любую трату — на материалы или работу мастеров. Полный контроль и никаких неожиданных списаний.

### Фото и видеофиксация

Забудьте о необходимости ежедневно приезжать на объект. Мы ежедневно фиксируем каждый этап работ на фото и видео. Вы в реальном времени можете наблюдать за прогресс

In [20]:
parser._remove_noise(soup)
main_node = parser._extract_main_node(soup)

print(len(main_node.select(".title_h1")))
print(len(main_node.select(".title_h2")))

3
7


In [12]:
for tag in main_node.select(".title_h1")[:10]:
    print("TITLE_H1:", tag.get_text(" ", strip=True))

for tag in main_node.select(".title_h2")[:10]:
    print("TITLE_H2:", tag.get_text(" ", strip=True))

TITLE_H1: Посмотрите последние наши реализованные объекты
TITLE_H1: Наши преимущества
TITLE_H1: Виды ремонта
TITLE_H2: От 15000₽/м²
TITLE_H2: Прозрачный учет расходов
TITLE_H2: Фото и видеофиксация
TITLE_H2: Гарантия результата
TITLE_H2: Увидеть своими глазами
TITLE_H2: Полный цикл работ
TITLE_H2: Собственный склад материалов


In [13]:
headings = parser._extract_headings(main_node)
print(headings[:20])

['Ремонт квартир и дизайн интерьера в Москве', 'Косметический ремонт', 'Капитальный ремонт', 'Дизайнерский ремонт']


In [14]:
for tag in main_node.find_all(["div"], recursive=True):
    classes = tag.get("class", [])
    if "title_h1" in classes or "title_h2" in classes:
        print(classes, tag.get_text(" ", strip=True))

['title_h2'] От 15000₽/м²
['title_h1'] Посмотрите последние наши реализованные объекты
['title_h1'] Наши преимущества
['title_h2'] Прозрачный учет расходов
['title_h2'] Фото и видеофиксация
['title_h2'] Гарантия результата
['title_h2'] Увидеть своими глазами
['title_h2'] Полный цикл работ
['title_h1'] Виды ремонта
['title_h2'] Собственный склад материалов


## another page

In [30]:
url2 = "https://sk-family.ru/zakazat-remont-kvartir-pod-kluch-v-moskve/"
response2 = requests.get(url2, timeout=20)
html2 = response2.text

In [31]:
doc2 = parser.parse_page(url2, html2)

In [33]:
print(doc2.title)
print(doc2.h1)
print(doc2.headings[:10])
print(doc2.text[:3000])
print(doc2.metadata)

СК Family - заказать ремонт квартиры под ключ в Москве
Заказать ремонт квартиры под ключ в Москве в СК Family
['Заказать ремонт квартиры под ключ в Москве в СК Family', 'Ответьте на 5 вопросов, и получите расчет стоимости ремонта квартиры под ключ в Москве', 'Заказать ремонт квартиры под ключ в Москве с СК Family', 'Видео обзоры', 'Посмотрите последние наши работы']
# Заказать ремонт квартиры под ключ в Москве в СК Family

Сделаем ремонт квартиры под ключ в Москве под ключ – без задержек и переплат!

Официальный договор

Фиксированная смета

Гарантия до 3 лет

Оплата по факту

## Видео обзоры

## Ответьте на 5 вопросов, и получите расчет стоимости ремонта квартиры под ключ в Москве

Свяжемся удобным способом в WhatsApp, Telegram или по телефону — без спама и лишних звонков

## Посмотрите последние наши работы

### Заказать ремонт квартиры под ключ в Москве с СК Family

Если вы хотите преобразить свое жилье и ищете, где заказать ремонт квартиры под ключ в Москве, компания СК Family — эт

In [34]:
url3 = "https://sk-family.ru/portfolio/zhk-foriver/"
response3 = requests.get(url3, timeout=20)
html3 = response3.text

In [35]:
doc2 = parser.parse_page(url3, html3)

In [36]:
print(doc2.title)
print(doc2.h1)
print(doc2.headings[:10])
print(doc2.text[:3000])
print(doc2.metadata)

Дизайн интерьера квартиры в ЖК Foriver
Дизайн интерьера квартиры в ЖК Foriver от СК Family: Ваш идеальный дом начинается здесь
['Дизайн интерьера квартиры в ЖК Foriver от СК Family: Ваш идеальный дом начинается здесь']
ул. Эдуарда Стрельцова, 2к1

# Дизайн интерьера квартиры в ЖК Foriver от СК Family: Ваш идеальный дом начинается здесь

Ищете вдохновение для создания уникального и комфортного интерьера своей квартиры в ЖК Foriver? СК Family предлагает профессиональные услуги по дизайну интерьера, которые помогут вам превратить ваше жилье в уютное и стильное пространство, отражающее вашу индивидуальность.

Почему выбирают ЖК Foriver?

Жилой комплекс Foriver — это не просто квартиры, это целая концепция комфортной жизни. Расположенный в живописном районе, Foriver предлагает идеальные условия для проживания, где природа и городская инфраструктура гармонично сосуществуют. Мы понимаем, что ваш дом должен быть отражением вашего стиля и потребностей, и именно поэтому предлагаем уникальные реш

# Crawler tests

In [65]:
crawler = SitemapCrawler(
    sitemap_index_url="https://sk-family.ru/sitemap_index.xml"
)

In [39]:
xml_text = crawler.fetch_xml("https://sk-family.ru/sitemap_index.xml")
print(xml_text[:500])

<?xml version="1.0" encoding="UTF-8"?><?xml-stylesheet type="text/xsl" href="//sk-family.ru/wp-content/plugins/wordpress-seo/css/main-sitemap.xsl"?>
<sitemapindex xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">
	<sitemap>
		<loc>https://sk-family.ru/post-sitemap.xml</loc>
		<lastmod>2025-01-28T14:13:32+00:00</lastmod>
	</sitemap>
	<sitemap>
		<loc>https://sk-family.ru/page-sitemap.xml</loc>
		<lastmod>2026-04-23T14:02:07+00:00</lastmod>
	</sitemap>
	<sitemap>
		<loc>https://sk-family.ru/us_


In [40]:
sitemap_urls = crawler.parse_sitemap_index(xml_text)
print(sitemap_urls)
print(len(sitemap_urls))

['https://sk-family.ru/post-sitemap.xml', 'https://sk-family.ru/page-sitemap.xml', 'https://sk-family.ru/us_portfolio-sitemap.xml', 'https://sk-family.ru/quizle-sitemap.xml', 'https://sk-family.ru/category-sitemap.xml', 'https://sk-family.ru/us_portfolio_category-sitemap.xml', 'https://sk-family.ru/author-sitemap.xml']
7


In [41]:
filtered = crawler.filter_sitemaps(sitemap_urls)
print(filtered)

['https://sk-family.ru/post-sitemap.xml', 'https://sk-family.ru/page-sitemap.xml', 'https://sk-family.ru/us_portfolio-sitemap.xml']


In [43]:
page_xml = crawler.fetch_xml(filtered[1])
page_urls = crawler.parse_url_sitemap(page_xml)

print(len(page_urls))
print(page_urls[:10])

140
['https://sk-family.ru/', 'https://sk-family.ru/%d0%ba%d0%b0%d0%bb%d1%8c%d0%ba%d1%83%d1%80%d1%8f%d1%82%d0%be%d1%80/', 'https://sk-family.ru/thankyou/', 'https://sk-family.ru/quiz/', 'https://sk-family.ru/remont-trehkomnatnoy-kvartiry-nedorogo/', 'https://sk-family.ru/premium-remont-kvartir-pod-kluch/', 'https://sk-family.ru/remont-dvuhkomantnoy-kvartiry-nedorogo/', 'https://sk-family.ru/remont-dvukhkomnatnoy-kvartiry-pod-kluch/', 'https://sk-family.ru/remont-kvartiry-v-novostroyke-nedorogo/', 'https://sk-family.ru/remont-odnokomnatnoy-kvartiry-v-novostroyke/']


In [44]:
all_urls = crawler.discover_urls()
print(len(all_urls))
print(all_urls[:20])

217
['https://sk-family.ru/2025/01/28/kak-vibrat-podryadchika/', 'https://sk-family.ru/', 'https://sk-family.ru/%d0%ba%d0%b0%d0%bb%d1%8c%d0%ba%d1%83%d1%80%d1%8f%d1%82%d0%be%d1%80/', 'https://sk-family.ru/thankyou/', 'https://sk-family.ru/quiz/', 'https://sk-family.ru/remont-trehkomnatnoy-kvartiry-nedorogo/', 'https://sk-family.ru/premium-remont-kvartir-pod-kluch/', 'https://sk-family.ru/remont-dvuhkomantnoy-kvartiry-nedorogo/', 'https://sk-family.ru/remont-dvukhkomnatnoy-kvartiry-pod-kluch/', 'https://sk-family.ru/remont-kvartiry-v-novostroyke-nedorogo/', 'https://sk-family.ru/remont-odnokomnatnoy-kvartiry-v-novostroyke/', 'https://sk-family.ru/remont-odnokomnatnoy-kvartiry-pod-kluch/', 'https://sk-family.ru/remont-chetyrekhkomnatnoy-kvartiry-pod-kluch/', 'https://sk-family.ru/elitniy-remont-kvartir-pod-kluch/', 'https://sk-family.ru/remont-trekhkomnatnoy-kvartiry-pod-kluch/', 'https://sk-family.ru/remont-kvartir-premium-klassa/', 'https://sk-family.ru/remont-dvuhkomnatnoi-kvartiry-v-n

In [66]:
all_urls = crawler.discover_urls()
print(len(all_urls))
print(all_urls[:30])

214
['https://sk-family.ru/2025/01/28/kak-vibrat-podryadchika/', 'https://sk-family.ru/', 'https://sk-family.ru/remont-trehkomnatnoy-kvartiry-nedorogo/', 'https://sk-family.ru/premium-remont-kvartir-pod-kluch/', 'https://sk-family.ru/remont-dvuhkomantnoy-kvartiry-nedorogo/', 'https://sk-family.ru/remont-dvukhkomnatnoy-kvartiry-pod-kluch/', 'https://sk-family.ru/remont-kvartiry-v-novostroyke-nedorogo/', 'https://sk-family.ru/remont-odnokomnatnoy-kvartiry-v-novostroyke/', 'https://sk-family.ru/remont-odnokomnatnoy-kvartiry-pod-kluch/', 'https://sk-family.ru/remont-chetyrekhkomnatnoy-kvartiry-pod-kluch/', 'https://sk-family.ru/elitniy-remont-kvartir-pod-kluch/', 'https://sk-family.ru/remont-trekhkomnatnoy-kvartiry-pod-kluch/', 'https://sk-family.ru/remont-kvartir-premium-klassa/', 'https://sk-family.ru/remont-dvuhkomnatnoi-kvartiry-v-novovstrojke/', 'https://sk-family.ru/remont-kvartir-vo-vtorichke/', 'https://sk-family.ru/remont-trehkomnatnoj-kvartiry-v-novovstrojke/', 'https://sk-family

# IngestionPipeline tests

In [68]:
pipeline = IngestionPipeline(
    sitemap_index_url="https://sk-family.ru/sitemap_index.xml",
    parser=parser,
)

In [69]:
docs = pipeline.collect_documents()

In [70]:
print(len(docs))
print(docs[0].title)
print(docs[0].h1)
print(docs[0].metadata)
print(docs[0].text[:2000])

198
Как выбрать подрядчика для ремонта квартиры: советы и рекомендации - Ремонт квартир
Как выбрать подрядчика для ремонта квартиры: советы и рекомендации
{'url': 'https://sk-family.ru/2025/01/28/kak-vibrat-podryadchika/', 'title': 'Как выбрать подрядчика для ремонта квартиры: советы и рекомендации - Ремонт квартир', 'h1': 'Как выбрать подрядчика для ремонта квартиры: советы и рекомендации', 'headings': ['Как выбрать подрядчика для ремонта квартиры: советы и рекомендации'], 'source': 'sk-family.ru', 'text_length': 2768}
# Как выбрать подрядчика для ремонта квартиры: советы и рекомендации

Ремонт квартиры — это важный шаг, который требует внимательного подхода. Подрядчик играет ключевую роль в этом процессе. Как же выбрать надежного специалиста? Вот несколько советов от компании СК Family.

1. Определите свои потребности

Перед тем как искать подрядчика, четко определите, какие именно работы необходимо выполнить. Это могут быть косметические изменения, капитальный ремонт или перепланиро